In [115]:
import sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
import shap

# np.random.seed(1)

In [116]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import shap

#getting data
df = pd.read_csv('C:/Users/micha/OneDrive/Bureaublad/XAI/final/diabetes.csv')

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

feature_names = X.columns.tolist()

#splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=7,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#train the model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=7
)

model.fit(X_train_scaled, y_train)

explainer = shap.TreeExplainer(model)


def get_positive_class_shap(shap_values):
    """
    Return a 1D array with one SHAP value per feature.
    """
    return shap_values[0, :, 1]





In [ ]:

n_samples = min(50, len(X_test_scaled))
noise_std = 0.05

results = []

for i in range(n_samples):

    x = X_test_scaled[i].copy()
    #original sample
    pred_original = model.predict_proba(x.reshape(1, -1))[0, 1]

    shap_values = explainer.shap_values(x.reshape(1, -1))
    original_attr = get_positive_class_shap(shap_values)


    #randomly changed sample
    noise = np.random.normal(0, noise_std, size=x.shape)
    x_perturbed = x + noise

    pred_perturbed = model.predict_proba(
        x_perturbed.reshape(1, -1)
    )[0, 1]

    #makes sure the difference is "small"
    if abs(pred_original - pred_perturbed) > 0.10:
        continue

    shap_values_perturbed = explainer.shap_values(
        x_perturbed.reshape(1, -1)
    )
    perturbed_attr = get_positive_class_shap(
        shap_values_perturbed
    )
    #bnelow are different ways to measure robustness

    #spearman rank correlation
    rho, _ = spearmanr(original_attr, perturbed_attr)

    #mean absolute change in attribution values
    mean_change = np.mean(
        np.abs(original_attr - perturbed_attr)
    )

    #if the most important feature stayed the same
    top_original = feature_names[
        np.argmax(np.abs(original_attr))
    ]

    top_perturbed = feature_names[
        np.argmax(np.abs(perturbed_attr))
    ]

    same_top_feature = top_original == top_perturbed

    results.append({
        "sample": i,
        "prediction_before": pred_original,
        "prediction_after": pred_perturbed,
        "spearman_corr": rho,
        "mean_attr_change": mean_change,
        "top_feature_before": top_original,
        "top_feature_after": top_perturbed,
        "same_top_feature": same_top_feature
    })

#code to inspect my results
results_df = pd.DataFrame(results)

print(results_df.head())

print("\nOverall Robustness ")
print("Average Spearman correlation:",
      results_df["spearman_corr"].mean())

print("Average attribution change:",
      results_df["mean_attr_change"].mean())

print("Fraction with same top feature:",
      results_df["same_top_feature"].mean())



In [114]:
import torch
import torch.nn as nn
import torch.optim as optim
from captum.attr import IntegratedGradients

#getting data
df = pd.read_csv('C:/Users/micha/OneDrive/Bureaublad/XAI/final/diabetes.csv')

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

feature_names = X.columns.tolist()

#splitting data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=7,
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#make the scaled data into tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(
    y_train.values.reshape(-1, 1),
    dtype=torch.float32
)

#down here i build a neural network
class DiabetesNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

model = DiabetesNN(X_train_tensor.shape[1])

#training the neural network
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()


ig = IntegratedGradients(model)

#mean of training data
baseline = X_train_tensor.mean(dim=0).unsqueeze(0)

#testing robustness
n_samples = min(50, len(X_test_tensor))
noise_std = 0.05

results = []

all_feature_changes = []

for i in range(n_samples):

    x = X_test_tensor[i].unsqueeze(0)

    #original prediction
    pred_original = model(x).item()

    #original explanation
    attr_original, delta = ig.attribute(
        x,
        baselines=baseline,
        return_convergence_delta=True
    )

    original_attr = attr_original.detach().numpy()[0]

    #randomly changed input
    noise = np.random.normal(0, noise_std, size=x.shape)

    x_perturbed = x.detach().numpy() + noise
    x_perturbed = torch.tensor(x_perturbed, dtype=torch.float32)

    #prediction from that changed input
    pred_perturbed = model(x_perturbed).item()

    #only compare if perbutation is small enoguh
    if abs(pred_original - pred_perturbed) > 0.10:
        continue

    #change explanation
    attr_perturbed, delta = ig.attribute(
        x_perturbed,
        baselines=baseline,
        return_convergence_delta=True
    )

    perturbed_attr = attr_perturbed.detach().numpy()[0]

    #robustness metrics (again)
    rho, _ = spearmanr(original_attr, perturbed_attr)

    mean_change = np.mean(
        np.abs(original_attr - perturbed_attr)
    )

    top_original = feature_names[
        np.argmax(np.abs(original_attr))
    ]

    top_perturbed = feature_names[
        np.argmax(np.abs(perturbed_attr))
    ]

    same_top_feature = top_original == top_perturbed

    results.append({
        "sample": i,
        "prediction_before": pred_original,
        "prediction_after": pred_perturbed,
        "spearman_corr": rho,
        "mean_attr_change": mean_change,
        "top_feature_before": top_original,
        "top_feature_after": top_perturbed,
        "same_top_feature": same_top_feature
    })

    all_feature_changes.append(
        np.abs(original_attr - perturbed_attr)
    )


results_df = pd.DataFrame(results)

print(results_df.head())

print("\nIntegrated Gradients Robustness ")
print("Average Spearman correlation:",
      results_df["spearman_corr"].mean())

print("Average attribution change:",
      results_df["mean_attr_change"].mean())

print("Fraction with same top feature:",
      results_df["same_top_feature"].mean())



   sample  prediction_before  prediction_after  spearman_corr  \
0       0           0.049976          0.048451       1.000000   
1       1           0.814764          0.813144       1.000000   
2       2           0.775325          0.761660       1.000000   
3       3           0.079947          0.087053       0.928571   
4       4           0.360965          0.356007       0.952381   

   mean_attr_change top_feature_before top_feature_after  same_top_feature  
0          0.001998            Glucose           Glucose              True  
1          0.003108            Glucose           Glucose              True  
2          0.004152            Glucose           Glucose              True  
3          0.002992            Glucose           Glucose              True  
4          0.003150                BMI               BMI              True  

Integrated Gradients Robustness 
Average Spearman correlation: 0.9785714285714286
Average attribution change: 0.004035066
Fraction with same top f